# Chain-of-Thought Faithfulness — CS60075 Assignment 2, Project 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chimankarparag/cs60075-cot-faithfulness/blob/main/code/CoT_Faithfulness_Colab.ipynb)

**Parag Chimankar (23CS10049)** · IIT Kharagpur · CS60075 NLP

End-to-end notebook for Tasks 1, 2 and 3 on BIG-Bench-Hard logical deduction (five objects), with `meta-llama/Llama-3.1-8B-Instruct`:

1. **Task 1** — direct (no-CoT) answering baseline, zero-shot and 4-shot
2. **Task 2** — CoT faithfulness interventions (Lanham et al., 2023): early answering, adding a mistake, paraphrasing, filler tokens
3. **Task 3** — faithfulness-weighted self-consistency (5 CoTs per question, 50% early answering as the weight), tuned on held-out items and repeated over four seeds
4. **Model comparison** — the same pipeline with `Qwen/Qwen3-8B`

All logic lives in `code/common`, `code/task1`, `code/task2` and `code/task3`; this notebook only calls those scripts, so the notebook and the command line always give the same numbers.

## How to run

**In VS Code (Google Colab extension):** open this file → *Select Kernel* → *Colab* → sign in → pick a GPU runtime.
**In the browser:** click the badge above → *Runtime → Change runtime type → GPU*.

The Colab runtime cannot see files on your laptop, so the setup cell clones the repo from GitHub.

| Setting | Meaning |
| --- | --- |
| `DATASET` | Fixed to `bbh` (BIG-Bench-Hard logical deduction, five objects) — the group's assigned dataset |
| `REUSE_RESULTS` | `True`: rebuild every table and figure from the generations committed in `results/` (a few seconds, no GPU needed). `False`: regenerate everything with the model |
| `N_LIMIT` | `None` for the full sample, or e.g. `10` for a quick live test (written to `results/smoke/`) |

**Secrets** (Colab key icon, or environment variables): `HF_TOKEN` — a Hugging Face token whose account has accepted the Llama 3.1 licence (only for `REUSE_RESULTS = False`); `GITHUB_TOKEN` — only while the repo is private.

**GPU note:** a free T4 (~15 GB) cannot hold the 8B model in 16-bit, so on a T4 the model is loaded in 4-bit NF4 with transformers; on an L4/A100 it runs in bf16. The committed results were produced in bf16 with vLLM on the IIT KGP GPU servers; 4-bit greedy outputs can differ slightly.

In [ ]:
# ---- Settings -------------------------------------------------------------
DATASET = "bbh"        # BIG-Bench-Hard, logical deduction (five objects)
REUSE_RESULTS = True   # True = recompute metrics from committed generations; False = run the model
N_LIMIT = None         # e.g. 10 for a quick live test
REPO_URL = "https://github.com/Chimankarparag/cs60075-cot-faithfulness.git"

## 0. Setup

Clones the repo (or uses the local copy when the notebook is opened from inside it), installs the missing packages when the model will run, and logs in to Hugging Face.

In [ ]:
import os, sys, subprocess
from pathlib import Path

def get_secret(name):
    """Colab secret if available, else environment variable, else None."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

# Use the local repo if this notebook lives inside it (e.g. VS Code on a laptop or the submission zip).
here = Path.cwd()
repo = next((p for p in [here, *here.parents] if (p / "code" / "common").is_dir()), None)
if repo is None:
    token = get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url, "cs60075-cot-faithfulness"], check=True)
    repo = here / "cs60075-cot-faithfulness"
os.chdir(repo)
print("repo:", repo)

In [ ]:
if not REUSE_RESULTS:
    # Keep Colab's own torch (the cluster's torch 2.13+cu130 needs a newer driver than Colab has).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.17.0", "accelerate==1.15.0", "bitsandbytes==0.50.2",
                    "huggingface_hub==1.33.0", "statsmodels==0.15.0"], check=True)
    from huggingface_hub import login
    login(token=get_secret("HF_TOKEN"))

# Download (pinned revisions, checksum-verified) and prepare the evaluation sample.
!python code/common/data.py --dataset {DATASET}

In [ ]:
# Model settings for live runs: bf16 if the GPU has >= 22 GB, else 4-bit NF4.
MODEL_ARGS = ""
if not REUSE_RESULTS:
    import torch
    assert torch.cuda.is_available(), "Select a GPU runtime first"
    gib = torch.cuda.get_device_properties(0).total_memory / 2**30
    MODEL_ARGS = "--backend hf --batch-size 16" + ("" if gib >= 22 else " --quant 4bit")
    print(f"{torch.cuda.get_device_name(0)}: {gib:.0f} GiB -> {MODEL_ARGS}")
LIMIT_ARGS = f"--n-limit {N_LIMIT}" if N_LIMIT else ""
RESULTS = Path("results/smoke" if N_LIMIT else "results")

from IPython.display import Markdown, Image, display

## 1. Task 1 — Direct answering baseline (30 marks)

Greedy decoding, at most 16 new tokens, reply prefilled with "The answer is". Zero-shot, and 4-shot with the same four fixed exemplars (answers only). **Reports:** accuracy with 95% bootstrap CI, parse failures, outputs that hit the token cap.

In [ ]:
if REUSE_RESULTS:
    !python code/task1/run_direct.py --dataset {DATASET} --rescore > /dev/null
else:
    !python code/task1/run_direct.py --dataset {DATASET} {MODEL_ARGS} {LIMIT_ARGS}
display(Markdown((RESULTS / "task1" / DATASET / "summary.md").read_text()))

## 2. Task 2 — CoT faithfulness interventions (30 marks)

1. **Standard CoT** — "Let's think step by step:", T = 0.7, top-p 0.95, 512 tokens; its forced answer A0 and the accuracy gain over Task 1.
2. **Early answering** — answer after every prefix of the CoT; the score uses the 20/40/60/80 % points.
3. **Adding a mistake** — one content sentence rewritten with an error, rest of the CoT regenerated.
4. **Paraphrasing** — the CoT reworded without the question, checked to keep its numbers and conclusion.
5. **Filler tokens** — the CoT replaced by `" ..."` of the same token length.

Faithfulness score = share of items whose answer changes (early answering, mistake, filler) or stays the same (paraphrase) relative to A0.

In [ ]:
if REUSE_RESULTS:
    !python code/task2/run_task2.py --dataset {DATASET} --stages score > /dev/null
else:
    !python code/task2/run_task2.py --dataset {DATASET} {MODEL_ARGS} {LIMIT_ARGS}
out = RESULTS / "task2" / DATASET
display(Markdown((out / "summary.md").read_text()))

In [ ]:
display(Image(str(out / "figures" / "early_answering_curve.png"), width=520))
display(Image(str(out / "figures" / "faithfulness_scores.png"), width=520))

### Qualitative examples

`results/task2/bbh/examples.md` lists, for every intervention, items where the answer changed and where it did not, plus CoTs whose own stated answer disagrees with the forced answer. The first part is shown below.

In [ ]:
text = (out / "examples.md").read_text()
display(Markdown(text[:6000] + ("\n\n*[truncated — open the file for the rest]*" if len(text) > 6000 else "")))

## 3. Task 3 — Faithfulness-weighted self-consistency (40 marks)

1. **Sample** 5 CoTs per question (T 0.7) and read each one's final answer.
2. **Early-answer** every sample at 50% of its sentences; a sample counts as faithful (load-bearing) if the answer does not change. A stricter rule, *loadbearing*, also requires the final answer to differ from the model's no-reasoning answer (Lanham et al., Sec. 3).
3. **Vote** over the 5 answers: faithful samples weigh 1, the others alpha. Three fixed pipelines (unchanged with alpha = 0.5; the opposite flag direction as an ablation; plain self-consistency, alpha = 1) and a **tuned** one whose rule and alpha were chosen on the 96 BBH items that are neither evaluated nor exemplars (`--split dev`), never on the 150 evaluation items.
4. **Compare** accuracy with Tasks 1 and 2.
5. **Intervene**: run Task 2's four interventions on the CoT the vote selects and compare the aggregate faithfulness with Task 2.

The pipeline is repeated with sampling seeds 42, 20, 30 and 40 (`--seed N`); the last cell of this section summarises the four runs. Needs the Task 1 and Task 2 results above. A live run takes much longer than Task 2.

In [ ]:
if REUSE_RESULTS:
    !python code/task3/run_task3.py --dataset {DATASET} --stages score > /dev/null
else:
    !python code/task3/run_task3.py --dataset {DATASET} --split dev {MODEL_ARGS} {LIMIT_ARGS}   # choose the tuned rule
    !python code/task3/run_task3.py --dataset {DATASET} {MODEL_ARGS} {LIMIT_ARGS}
out3 = RESULTS / "task3" / DATASET
if (out3 / "dev" / "tuning.md").exists():
    display(Markdown((out3 / "dev" / "tuning.md").read_text()))
display(Markdown((out3 / "summary.md").read_text()))

In [ ]:
display(Image(str(out3 / "figures" / "accuracy_comparison.png"), width=520))
display(Image(str(out3 / "figures" / "faithfulness_comparison.png"), width=520))

### Task 3 across sampling seeds

Seeds 20, 30 and 40 are only replayed here (a live run of all four seeds takes hours; on the cluster use `scripts/cluster/run_campaign.sh`).

In [ ]:
if REUSE_RESULTS:
    for s in (20, 30, 40):
        if (out3 / f"seed_{s}" / "samples.jsonl").exists():
            !python code/task3/run_task3.py --dataset {DATASET} --seed {s} --stages score > /dev/null
    !python code/task3/seeds.py --dataset {DATASET} > /dev/null
if (out3 / "seeds_summary.md").exists():
    display(Markdown((out3 / "seeds_summary.md").read_text()))

## 4. The same pipeline with a stronger model

Every task was also run with `Qwen/Qwen3-8B` (thinking mode off) via `--model qwen3-8b`; its results are in `results/models/qwen3-8b/`. The table compares both models and lists where each run ran and how long it took (from the `run_info.json` files).

In [ ]:
!python code/analysis/compare_models.py > /dev/null
display(Markdown((RESULTS / "comparison" / "models.md").read_text()))
display(Image(str(RESULTS / "comparison" / "figures" / "models_accuracy.png"), width=620))